# 🧠 3D Multimodal JEPA Representation Learning & Volumetric Segmentation Benchmark
### Complete Kaggle GPU Training, Baselines, Fine-Tuning, Probing & Evaluation Runner

This standalone Kaggle notebook provides a complete execution pipeline for evaluating Self-Supervised Learning (SSL) architectures on 3D volumetric multi-modal MRI scans (**BraTS 2024 GLI**).

**Architectures evaluated:**
- **3D SigReg JEPA**: Single-encoder with Epps-Pulley empirical characteristic function Gaussianity regularization over random 1D projections (Balestriero & LeCun, 2025).
- **3D VisReg JEPA**: Single-encoder with decoupled Center, Scale (two-sided variance penalty), and 1D Sliced-Wasserstein Distance Shape regularization (Wu, Balestriero, & Levine, 2026).
- **3D I-JEPA**: Dual-encoder with EMA teacher target regularization (Assran et al., CVPR 2023).
- **3D Residual UNet**: Classical Supervised 3D Baseline (MONAI; Ronneberger et al., 2015; Milletari et al., 2016).
- **3D nnU-Net**: Supervised SOTA MONAI DynUNet with Deep Supervision (Isensee et al., Nature Methods 2021).

---
### ⚡ Hardware & Acceleration Recommendations
- **Accelerator:** GPU T4 x 1 or GPU T4 x 2 (Tesla T4, 16GB VRAM each) or P100.
- **Mixed Precision (AMP):** Enabled by default via `--amp` (`torch.amp.autocast('cuda')` + `GradScaler`) for ~3.2x training speedup and 50% lower VRAM usage.
- **Internet Access:** Turn **ON** (in right sidebar: Settings -> Internet -> Turn On) to install dependencies.
- **Persistence:** All checkpoints, logs, and benchmark summaries are saved to `/kaggle/working/outputs/` and compressed to `outputs.zip` for 1-click download.

## 1. Hardware & CUDA Environment Verification
Verify GPU allocation and CUDA driver state.

In [ ]:
!nvidia-smi

import torch

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Device Count:     {torch.cuda.device_count()}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Please navigate to Notebook Settings -> Accelerator -> GPU T4!"
    )

## 2. Dependencies Installation
Install `monai`, `nibabel`, and `tabulate`.

In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")

## 3. Codebase Setup & Editable Installation
Clones your GitHub repository or sets up the local package in editable mode.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

# Setup working directory in /kaggle/working
REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/kaggle/working/tumor_segmentation_3d")

# Clean up broken or incomplete clone from previous failed runs
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

thesis_repo = Path("/kaggle/working/thesis_repo")
if thesis_repo.exists() and not (thesis_repo / "src" / "brats_jepa_3d").exists():
    shutil.rmtree(thesis_repo)

# Clone repository if not already present
if not (work_dir / "src" / "brats_jepa_3d").exists():
    if (thesis_repo / "src" / "brats_jepa_3d").exists():
        work_dir = thesis_repo
    elif Path("/kaggle/working/src/brats_jepa_3d").exists():
        work_dir = Path("/kaggle/working")
    else:
        print(f"Cloning codebase from: {REPO_URL} ...")
        !git clone {REPO_URL} {work_dir}

# Verify package was successfully cloned
src_dir = work_dir / "src"
if not (src_dir / "brats_jepa_3d").exists():
    raise RuntimeError(
        "❌ Clone failed! The package 'brats_jepa_3d' was not found on disk.\n"
        "👉 Please ensure 'Internet' is toggled ON in the Kaggle notebook settings (right sidebar)!"
    )

# Change working directory and update sys.path
os.chdir(str(work_dir))
%cd {work_dir}

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Install in editable mode
!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"


## 4. Dataset Discovery & Health Checks
Verifies whether processed `.npz` volumes exist. If absent, automatically invokes `prepare_data_3d.py` with multi-worker parallel resampling (`--num_workers 4`) and compact `float16` storage (~6.7 MB per volume, upcast to FP32 in RAM upon loading).


In [ ]:
import pandas as pd
import torch

from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

print("=== DATASET DISCOVERY ===")
data_dir = get_dataset_dir("brats_gli_3d")
meta_path = get_metadata_path("brats_gli_3d")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

# If preprocessed dataset is not found, check for raw BraTS data and run prepare_data_3d.py
if not meta_path.exists():
    print("\n⚠️ Preprocessed dataset not found. Running 3D preprocessing from raw BraTS data...")
    !python scripts/prepare_data_3d.py --limit 100 --dtype float16 --num_workers 4
    meta_path = get_metadata_path("brats_gli_3d")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())

    ds = BraTS3DDataset(split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    print(
        "❌ ERROR: Please attach 'brats-3d-datasets' or raw BraTS dataset via '+ Add Input' in Kaggle!"
    )


## 5. Self-Supervised JEPA Pre-training (50 Epochs, AMP)
Trains **3D SigReg JEPA** (Epps-Pulley Gaussianity test), **3D VisReg JEPA** (Decoupled Center, Scale unit variance, and Sliced-Wasserstein Shape), and standard **3D I-JEPA** on 4-channel 3D MRI volumes.

- **I-JEPA EMA Momentum**: Dynamically annealed via cosine schedule $m_t = m_{\text{end}} - (m_{\text{end}} - m_{\text{start}}) \cdot \frac{1}{2}(1 + \cos(\pi t / T_{\text{anneal}}))$ with $m \in [0.996, 1.0]$.
- **Data & Masking Transforms**: Loaded automatically from `configs/dataset/brats3d.yaml` (context/target block scale ranges, aspect ratio bounds, volumetric augmentations).

> *Note: To run a rapid dry-run test, append `--smoke_test` or `--epochs 2`.*


In [ ]:
# 5.1 3D SigReg JEPA Pre-training (Epps-Pulley Gaussianity Test, Scale Factor N)
!python scripts/train_jepa_3d.py \
    --model_type sigreg_jepa \
    --epochs 50 \
    --batch_size 4 \
    --amp

In [ ]:
# 5.2 3D VisReg JEPA Pre-training (Decoupled Center, Scale, Shape SWD)
!python scripts/train_jepa_3d.py \
    --model_type visreg_jepa \
    --epochs 50 \
    --batch_size 4 \
    --amp

In [ ]:
# 5.3 3D I-JEPA Pre-training (EMA Teacher Network)
!python scripts/train_jepa_3d.py \
    --model_type ijepa \
    --epochs 50 \
    --batch_size 4 \
    --amp

## 6. Supervised 3D Baselines (30 Epochs, AMP)
Trains supervised **3D Residual UNet** and state-of-the-art **3D nnU-Net** (DynUNet with multi-scale deep supervision weighted via dynamic exponential decay $w_s = 2^{-s} / \sum_{j=0}^{S-1} 2^{-j}$).

- Both baselines utilize `--batch_size 2` for optimal VRAM safety and numerical stability on 16GB GPUs (Tesla T4 / P100).


In [ ]:
# 6.1 3D Residual UNet Baseline (MONAI)
!python scripts/train_unet_3d.py \
    --epochs 30 \
    --batch_size 2 \
    --amp


In [ ]:
# 6.2 3D nnU-Net DynUNet with Deep Supervision (MONAI)
!python scripts/train_nnunet_3d.py \
    --epochs 30 \
    --batch_size 2 \
    --amp

## 7. Downstream 3D Volumetric Segmentation Fine-Tuning
Fine-tunes pre-trained JEPA encoders with the **Hierarchical Multi-Scale 3D FPN Decoder** ($L_2, L_4, L_6, L_8$ lateral skips).

In [ ]:
# 7.1 Fine-tune 3D SigReg JEPA with Multi-Scale FPN
!python scripts/train_downstream_3d.py \
    --model_type sigreg_jepa \
    --decoder_type multiscale \
    --epochs 30 \
    --batch_size 2 \
    --amp

In [ ]:
# 7.2 Fine-tune 3D VisReg JEPA with Multi-Scale FPN
!python scripts/train_downstream_3d.py \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --epochs 30 \
    --batch_size 2 \
    --amp

In [ ]:
# 7.3 Fine-tune 3D I-JEPA with Multi-Scale FPN
!python scripts/train_downstream_3d.py \
    --model_type ijepa \
    --decoder_type multiscale \
    --epochs 30 \
    --batch_size 2 \
    --amp

## 8. Master 3D Volumetric Benchmark Evaluation
Evaluates all models on the independent test split across:
- **3D Dice Score (%)** & **3D IoU (%)**
- **Exact 3D 95th Percentile Hausdorff Distance (HD95 mm)** via `scipy.spatial.cKDTree`
- **Inference Latency (ms / volume)**
- **Effective Rank ($S_k^2$)** & **Centered Cosine Similarity**

In [ ]:
!python scripts/evaluate_3d.py --batch_size 2 --amp

from IPython.display import Markdown, display

from brats_jepa_3d.config import METRICS_DIR

summary_md = METRICS_DIR / "benchmark_3d_summary.md"
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

## 9. Low-Data Label Efficiency Benchmark
Trains models across constrained annotation budgets ($1\%, 5\%, 10\%, 25\%, 50\%, 100\%$ labels) to assess representation transfer efficiency.

In [ ]:
!python scripts/evaluate_low_data_3d.py \
    --fractions 0.01 0.05 0.10 0.25 0.50 1.00 \
    --amp

from IPython.display import Markdown, display

from brats_jepa_3d.config import METRICS_DIR

low_data_md = METRICS_DIR / "low_data_3d_summary.md"
if low_data_md.exists():
    display(Markdown(low_data_md.read_text()))

## 10. Out-of-Distribution (OOD) Scanner Shift Robustness
Stress-tests models against simulated clinical MRI artifacts:
1. **3D Rician Scanner Noise** ($\sigma = 0.08$; Gudbjartsson & Patz, 1995)
2. **3D B1 RF Field Inhomogeneity** (2nd-order polynomial; Sled et al., 1998)
3. **Missing MRI Sequences** (T1c-only and FLAIR-only emergency triage)

In [ ]:
!python scripts/evaluate_ood_3d.py --amp

from IPython.display import Markdown, display

from brats_jepa_3d.config import METRICS_DIR

ood_md = METRICS_DIR / "ood_3d_summary.md"
if ood_md.exists():
    display(Markdown(ood_md.read_text()))

## 11. Multi-Planar Orthogonal Visualizations & Publication Figures
Generates high-resolution publication figures (Axial, Coronal, Sagittal slices, metric comparison bars, label efficiency curves, OOD robustness).

In [ ]:
!python scripts/generate_figures_3d.py

from IPython.display import Image, display

from brats_jepa_3d.config import FIGURES_DIR

for fig_name in [
    "orthogonal_multi_planar_figure.png",
    "benchmark_3d_comparison.png",
    "low_data_3d_efficiency.png",
    "ood_3d_robustness.png",
]:
    p = FIGURES_DIR / fig_name
    if p.exists():
        print(f"\n=== {fig_name} ===")
        display(Image(str(p)))

## 12. 1-Click Output Archival & Download
Archives all checkpoints, CSV/MD benchmark reports, log files, and vector PDF/PNG figures into a single `outputs.zip` file.

In [ ]:
import os
import zipfile
from pathlib import Path

from brats_jepa_3d.config import OUTPUTS_DIR

zip_path = Path("/kaggle/working/outputs.zip")
print(f"Archiving {OUTPUTS_DIR} -> {zip_path} ...")

count = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUTS_DIR):
        for f in files:
            fp = Path(root) / f
            zf.write(fp, arcname=fp.relative_to(OUTPUTS_DIR.parent))
            count += 1

print(f"✓ Archived {count} files. Zip size: {zip_path.stat().st_size / 1e6:.2f} MB")
print(f"📍 File ready for download: {zip_path}")